In [1]:
import torch
torch.__version__

'2.14.1'

In [2]:
# to set device
device = torch.device("mps") if torch.mps.is_available() else torch.device("cpu") # seems changes from backend.mps.is_available(), maybenot


In [3]:
zd = torch.tensor(0)
print(zd.shape)
oned = torch.tensor([0,1,2,3])
print(oned.shape)
twod = torch.tensor([[2,3], [0,1]])
print(twod.shape)

torch.Size([])
torch.Size([4])
torch.Size([2, 2])


In [4]:
import torch
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([0.1, 0.2, 0.3], requires_grad=True)
b = torch.tensor([0.1], requires_grad=True)

for _ in range(5):
    y_pred = torch.sigmoid(x @ w + b)
    
    loss = F.binary_cross_entropy(y_pred, y)
    with torch.no_grad():
        grads = grad(loss, [w,b])
        w -= 0.001 * grads[0]
        b -= 0.001 * grads[1]   
    print(loss)
    print(w)
    print(b)



tensor(0.2014, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.1002, 0.2004, 0.3005], requires_grad=True)
tensor([0.1002], requires_grad=True)
tensor(0.2009, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.1004, 0.2007, 0.3011], requires_grad=True)
tensor([0.1004], requires_grad=True)
tensor(0.2004, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.1005, 0.2011, 0.3016], requires_grad=True)
tensor([0.1005], requires_grad=True)
tensor(0.1999, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.1007, 0.2015, 0.3022], requires_grad=True)
tensor([0.1007], requires_grad=True)
tensor(0.1994, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.1009, 0.2018, 0.3027], requires_grad=True)
tensor([0.1009], requires_grad=True)


In [5]:
y = torch.tensor([1.0])
x = torch.tensor([2.0,-2.0,1.0])
w = torch.tensor([0.5,0.1,0.2], requires_grad=True)
b = torch.tensor([0.1], requires_grad=True)

for _ in range(5):
    y_pred = torch.sigmoid(x @ w + b)
    loss = F.binary_cross_entropy(y_pred, y)
    grads = grad(loss, [w,b])
    with torch.no_grad():
        w -= 0.01 * grads[0]
        b -= 0.01 * grads[1]
    print(loss)
    print(w)

tensor(0.2873, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.5050, 0.0950, 0.2025], requires_grad=True)
tensor(0.2812, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.5099, 0.0901, 0.2049], requires_grad=True)
tensor(0.2752, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.5147, 0.0853, 0.2074], requires_grad=True)
tensor(0.2695, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.5194, 0.0806, 0.2097], requires_grad=True)
tensor(0.2639, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([0.5241, 0.0759, 0.2120], requires_grad=True)


In [6]:
class MyNeuralNetwork(torch.nn.Module):

    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 5),
            torch.nn.ReLU(), 
            torch.nn.Linear(5, 3),
            torch.nn.ReLU(),
            torch.nn.Linear(3, num_outputs),
        )

    def forward(self, x):
        return self.layers(x)

model = MyNeuralNetwork(10, 2)

In [7]:
for p in model.parameters():
    print(p.numel())

print(model.layers[0].weight.shape)

50
5
15
3
6
2
torch.Size([5, 10])


In [8]:
X = torch.rand((1,10))
out = model(X)
torch.softmax(out, dim=1)

tensor([[0.5784, 0.4216]], grad_fn=<SoftmaxBackward0>)

In [9]:
# data set 

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0,0,0,1,1])

X_test = torch.tensor([
    [-1.0, 3.0],
    [2.5, -1.3]
])
y_test = torch.tensor([0,1])


In [10]:
from torch.utils.data import Dataset   
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return self.features.shape[0]

train_dataset = ToyDataset(X_train, y_train)
test_dataset = ToyDataset(X_test, y_test)
print(len(train_dataset))
print(len(test_dataset))

5
2


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(dataset = train_dataset, batch_size = 2,
                        shuffle=True, num_workers=0)
test_loader = DataLoader(dataset = test_dataset, batch_size = 2,
                        shuffle=False, num_workers=0)

for idx, (x, y) in enumerate(train_loader):
    print(f"batch {idx}, {x}, {y}")

batch 0, tensor([[-0.5000,  2.6000],
        [-0.9000,  2.9000]]), tensor([0, 0])
batch 1, tensor([[ 2.3000, -1.1000],
        [-1.2000,  3.1000]]), tensor([1, 0])
batch 2, tensor([[ 2.7000, -1.5000]]), tensor([1])


In [ ]:
# training loop 

model = MyNeuralNetwork(num_inputs = 2, num_outputs = 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for _ in range(5):
    model.train()

    for id, (x, y) in enumerate(train_loader):
        logits = model(x)
        loss = F.binary_cross_entropy_with_logits(logits.squeeze(1), y.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Batch {id:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()

ValueError: Target size (torch.Size([2])) must be the same as input size (torch.Size([2, 1]))